In [1]:
!hostname

gpue06.delta.ncsa.illinois.edu


In [2]:
import os, json, scanpy as sc, numpy as np, pandas as pd, anndata as ad, pyranges as pr
from pathlib import Path

ad.settings.allow_write_nullable_strings = True
pd.set_option("mode.string_storage", "python")

human_genome_path = "/work/hdd/bgdb/asachan/datasets_proj/human_genome_files"
out_tmp           = Path("/projects/bhdw/asachan/tmp")
out_zero_shot     = Path("/projects/bhdw/asachan/methods/maxtoki-perturb/data/zero_shot")
out_zero_shot.mkdir(parents=True, exist_ok=True)

vocab_path = "/projects/bhdw/asachan/methods/maxtoki-perturb/src/maxtoki_mlx/resources/token_dictionary.json"

### Load RNA data

In [3]:
adata_rna = sc.read_h5ad(
    "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/"
    "rna_objects/rna_female_type2_ds_wrt_HALLMARK_DNA_REPAIR.h5ad"
)
print(adata_rna)

AnnData object with n_obs × n_vars = 3989 × 48355
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
    var: 'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'neighbors', 'pca', 'rank_genes_groups', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'score_aucell'
    varm: 'PCs'
    layers: 'counts', 'counts_float', 'lognorm'
    obsp: 'connectivities', 'distances'


#### Map ENSG from gencode

In [4]:
gtf = pr.read_gtf(f"{human_genome_path}/gencode.v46.annotation.gtf").df
gtf = gtf[gtf.Feature == "gene"][["gene_name","gene_id"]].drop_duplicates("gene_name")
gtf["gene_id_clean"] = gtf["gene_id"].str.split(".").str[0]
sym2ensg = dict(zip(gtf["gene_name"].astype(str), gtf["gene_id_clean"]))
print(f"GTF mappings: {len(sym2ensg)} symbol→ENSG")

syms = adata_rna.var_names.astype(str)
adata_rna.var["ensg"] = [sym2ensg.get(s, None) for s in syms]
print(f"{adata_rna.var['ensg'].notna().sum()}/{adata_rna.n_vars} symbols mapped")

GTF mappings: 61471 symbol→ENSG
29718/48355 symbols mapped


### Maxtoki vocab subset of features
#### To improve inference time

In [5]:
with open(vocab_path) as f:
    vocab = json.load(f)
ensg_to_token = {k: v for k, v in vocab.items() if k.startswith("ENSG")}
print(f"vocab tokens: {len(vocab)} ({len(ensg_to_token)} ENSG-keyed)")

adata_rna.var["maxtoki_token"] = (
    adata_rna.var["ensg"].map(ensg_to_token).astype("Int64")
)

mask = adata_rna.var["maxtoki_token"].notna()
print(f"keeping {mask.sum()} / {adata_rna.n_vars} genes ({mask.sum()/adata_rna.n_vars:.1%})")

rna_sub = adata_rna[:, mask.values].copy()
rna_sub.var["maxtoki_token"] = rna_sub.var["maxtoki_token"].astype(int)

# sort genes by token id so column index matches vocab order
rna_sub = rna_sub[:, np.argsort(rna_sub.var["maxtoki_token"].values)].copy()
rna_sub.var["ensembl_id"] = rna_sub.var["ensg"]                   # MaxToki convention

print(rna_sub)

vocab tokens: 20277 (20271 ENSG-keyed)
keeping 18470 / 48355 genes (38.2%)
AnnData object with n_obs × n_vars = 3989 × 18470
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
    var: 'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensg', 'maxtoki_token', 'ensembl_id'
    uns: 'hvg', 'neighbors', 'pca', 'rank_genes_groups', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'score_aucell'
    varm: 'PCs'
    layers: 'counts', 'counts_float', 'lognorm'
    obsp: 'connectivities', 'distances'


In [6]:
print("vocab tokens with no expression in adata:",
      len(set(ensg_to_token.values()) - set(rna_sub.var["maxtoki_token"])))

# row-wise sanity: tokens are sorted, unique, and non-negative
tok = rna_sub.var["maxtoki_token"].values
assert (np.diff(tok) > 0).all() and (tok >= 4).all()
print("token order: monotone increasing, all ≥ 4 (special tokens are 0–3)")

vocab tokens with no expression in adata: 1801
token order: monotone increasing, all ≥ 4 (special tokens are 0–3)


In [7]:
rna_sub.write_h5ad(out_zero_shot / "rna_zero_shot_input.h5ad")
# also keep a copy in the temp dir so notebook B can pick it up without crossing dirs
rna_sub.write_h5ad(out_tmp / "rna_zero_shot_input.h5ad")
print("saved:", out_zero_shot / "rna_zero_shot_input.h5ad")

saved: /projects/bhdw/asachan/methods/maxtoki-perturb/data/zero_shot/rna_zero_shot_input.h5ad


### Check if the zero-shot input matches the multiome output

In [3]:
#load the multiome output
multiome_output = sc.read_h5ad("/projects/bhdw/asachan/tmp/atac_rna_pairing_skm_prep/rna_sub.h5ad")
#load the zero-shot input
zero_shot_input = sc.read_h5ad(out_tmp / "rna_zero_shot_input.h5ad")

In [5]:

multiome_output

AnnData object with n_obs × n_vars = 3989 × 18470
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
    var: 'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensg', 'maxtoki_token', 'ensembl_id'
    obsm: 'X_pca'

In [6]:
zero_shot_input

AnnData object with n_obs × n_vars = 3989 × 18470
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
    var: 'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensg', 'maxtoki_token', 'ensembl_id'
    uns: 'hvg', 'neighbors', 'pca', 'rank_genes_groups', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'score_aucell'
    varm: 'PCs'
    layers: 'counts', 'counts_float', 'lognorm'
    obsp: 'connectivities', 'distances'